In [2]:
import openai
print(openai.__version__)

2.15.0


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env
api_key = os.getenv("OPENAI_API_KEY")

# Verify API key is loaded (don't print the actual key)
if api_key:
    print(" OpenAI API key loaded successfully")
else:
    print("OpenAI API key not found in .env file")

In [5]:
import pandas as pd
import numpy as np
import requests

In [15]:
df = pd.read_csv("../data/milestone1_output_nadvik.csv")
df.head(2)

,contract_id,extracted_text,apr,term_months,monthly_payment,penalty
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,none
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,early termination fee $300


In [1]:
from openai import OpenAI
import os
import json
import re

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [17]:
# ================================
# Week 3 – LLM Prompt Design
# ================================

SLA_PROMPT_TEMPLATE = """
You are a legal contract analysis assistant specialized in car lease agreements.

Extract the following SLA details from the contract text.
If a value is missing, return null.
Return ONLY valid JSON. No explanation.

Fields:
- interest_rate_apr
- lease_term_months
- monthly_payment
- down_payment
- residual_value
- mileage_allowance
- overage_charge
- early_termination
- purchase_option
- maintenance_responsibility
- warranty_insurance
- penalties
- missing_or_ambiguous_clauses

Contract Text:
\"\"\"
{contract_text}
\"\"\"
"""

In [18]:
# ================================
# Week 3 – OpenAI LLM Extraction
# ================================



def extract_sla_with_gpt(contract_text):
    prompt = SLA_PROMPT_TEMPLATE.format(contract_text=contract_text)

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    raw_output = response.output_text.strip()

    match = re.search(r"\{[\s\S]*\}", raw_output)
    if not match:
        return {"error": "No JSON found", "raw_output": raw_output}

    try:
        return json.loads(match.group(0))
    except Exception as e:
        return {"error": str(e), "json_text": match.group(0)}

In [19]:
test_output = extract_sla_with_gpt(df.loc[0,
"extracted_text"])
test_output

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# ================================
# Week 3 – Apply LLM & Store SLA
# ================================

sample_df = df.head(1)

sla_json_records = sample_df.apply(
    lambda row: {
        "contract_id": int(row.name),
        "sla": extract_sla_with_gpt(row["extracted_text"])
    },
    axis=1
).tolist()

sla_json_records[0]

# Store SLA JSON
with open("../data/milestone2_sla_output.json", "w") as f:
    json.dump(sla_json_records, f, indent=2)

print("SLA JSON stored successfully using OpenAI GPT")


In [ ]:
# ===============================
# Week 3 – Accuracy Evaluation (Sample Only)
# ===============================

# Use only 1 sample contract to avoid rate limits
eval_df = df.head(1).copy()

# Apply LLM extraction (SAFE – only 1 call)
eval_df["llm_sla"] = eval_df["extracted_text"].apply(extract_sla_with_gpt)

eval_df[["llm_sla"]]


In [ ]:
# Extract individual SLA fields from LLM output
eval_df["llm_apr"] = eval_df["llm_sla"].apply(lambda x: x.get("interest_rate_apr"))
eval_df["llm_term"] = eval_df["llm_sla"].apply(lambda x: x.get("lease_term_months"))
eval_df["llm_payment"] = eval_df["llm_sla"].apply(lambda x: x.get("monthly_payment"))

eval_df[["llm_apr", "llm_term", "llm_payment"]]


{'ABS': '',
 'ActiveSafetySysNote': '',
 'AdaptiveCruiseControl': '',
 'AdaptiveDrivingBeam': '',
 'AdaptiveHeadlights': '',
 'AdditionalErrorText': 'The Model Year decoded for this VIN may be incorrect. If you know the Model year, please enter it and decode again to get more accurate information.',
 'AirBagLocCurtain': '',
 'AirBagLocFront': '1st Row (Driver and Passenger)',
 'AirBagLocKnee': '',
 'AirBagLocSeatCushion': '',
 'AirBagLocSide': '1st and 2nd Rows',
 'AutoReverseSystem': '',
 'AutomaticPedestrianAlertingSound': '',
 'AxleConfiguration': '',
 'Axles': '',
 'BasePrice': '',
 'BatteryA': '',
 'BatteryA_to': '',
 'BatteryCells': '',
 'BatteryInfo': '',
 'BatteryKWh': '',
 'BatteryKWh_to': '',
 'BatteryModules': '',
 'BatteryPacks': '',
 'BatteryType': '',
 'BatteryV': '',
 'BatteryV_to': '',
 'BedLengthIN': '',
 'BedType': '',
 'BlindSpotIntervention': '',
 'BlindSpotMon': '',
 'BodyCabType': 'Crew/Super Crew/Crew Max',
 'BodyClass': 'Pickup',
 'BrakeSystemDesc': '',
 'BrakeS

In [ ]:
# Ground truth values for sample contract (manual / known values)
eval_df["expected_apr"] = 10.49
eval_df["expected_term"] = 24
eval_df["expected_payment"] = 959

eval_df[[
    "llm_apr", "expected_apr",
    "llm_term", "expected_term",
    "llm_payment", "expected_payment"
]]



{'make': 'FORD', 'model': 'F-150', 'year': '2016', 'body_type': 'Pickup'}

In [ ]:
accuracy = {
    "APR Accuracy (%)": (eval_df["llm_apr"] == eval_df["expected_apr"]).mean() * 100,
    "Term Accuracy (%)": (eval_df["llm_term"] == eval_df["expected_term"]).mean() * 100,
    "Payment Accuracy (%)": (eval_df["llm_payment"] == eval_df["expected_payment"]).mean() * 100
}

accuracy

In [ ]:
# ===============================
# Week 4 – VIN Lookup (NHTSA API)
# ===============================

import requests

def fetch_vehicle_details_from_vin(vin):
    """
    Fetch vehicle make, model, year using NHTSA VIN Decode API
    """
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValuesExtended/{vin}?format=json"
    response = requests.get(url, timeout=10)
    data = response.json()

    if not data.get("Results"):
        return None

    result = data["Results"][0]

    return {
        "vin": vin,
        "make": result.get("Make"),
        "model": result.get("Model"),
        "year": result.get("ModelYear")
    }


{'make': 'FORD', 'model': 'F-150', 'year': '2016', 'body_type': 'Pickup'}

In [ ]:
def fetch_vehicle_recalls(make, model, year):
    """
    Fetch recall information using NHTSA Recall API
    """
    if not make or not model or not year:
        return []

    url = (
        "https://api.nhtsa.gov/recalls/recallsByVehicle"
        f"?make={make}&model={model}&modelYear={year}"
    )

    response = requests.get(url, timeout=10)
    data = response.json()

    recalls = data.get("results", [])

    return [
        {
            "campaign_number": r.get("NHTSACampaignNumber"),
            "summary": r.get("Summary"),
            "consequence": r.get("Consequence")
        }
        for r in recalls
    ]

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag,expected_apr,expected_term,expected_payment,expected_penalty,apr_score,term_score,payment_score,penalty_score,total_score,quality_score
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low,10.49,24,959,NaN,1,1,1,0,3,100.000000
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low,3.78,24,804,Early termination fee $300,1,1,1,1,4,133.333333
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High,5.41,24,774,Late fee $50,1,1,1,1,4,133.333333
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High,5.26,48,1028,Late fee $25,1,1,1,1,4,133.333333
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low,5.97,36,1180,NaN,1,1,1,0,3,100.000000


In [ ]:
test_vin = "1HGCM82633A004352"

vehicle_info = fetch_vehicle_details_from_vin(test_vin)
vehicle_info

In [ ]:
recalls = fetch_vehicle_recalls(
    vehicle_info["make"],
    vehicle_info["model"],
    vehicle_info["year"]
)

recalls[:2]

In [ ]:
# ===============================
# Week 4 – Combine SLA + Vehicle Data
# ===============================

def build_final_contract_response(contract_id, sla_data, vin):
    vehicle = fetch_vehicle_details_from_vin(vin)

    recalls = []
    if vehicle:
        recalls = fetch_vehicle_recalls(
            vehicle["make"],
            vehicle["model"],
            vehicle["year"]
        )

    return {
        "contract_id": contract_id,
        "sla": sla_data,
        "vehicle": vehicle,
        "recalls": recalls
    }

In [ ]:
# Internal end-to-end test

sample_contract = sla_json_records[0]
sample_vin = "1HGCM82633A004352"

final_response = build_final_contract_response(
    contract_id=sample_contract["contract_id"],
    sla_data=sample_contract["sla"],
    vin=sample_vin
)

final_response